In [ ]:
import sympy as sp
from IPython.display import display

sp.init_printing(use_unicode=True)

z = sp.Symbol('z', complex=True)
y = sp.Symbol('y')
n = sp.Symbol('n', integer=True, nonnegative=True)

X_z = (1 + sp.Rational(1, 4) * z**(-1)) / (1 - sp.Rational(1, 2) * z**(-1))**2
print("Original Z-transform X(z):")
display(X_z)

X_y = (1 + sp.Rational(1, 4) * y) / (1 - sp.Rational(1, 2) * y)**2
print("\nX(y), where y = z^(-1):")
display(X_y)

pfe_y = sp.apart(X_y, y)
print("\nPartial Fraction Expansion:")
display(pfe_y)

def table_lookup_inverse_z(term, y_var, n_var):
    u = sp.Heaviside(n_var)
    num, den = sp.fraction(sp.cancel(term))
    factors = sp.factor_list(den)
    
    for base, power in factors[1]:
        roots = sp.solve(base, y_var)
        if not roots:
            continue
        
        y0 = roots[0]
        a = sp.simplify(1 / y0)
        scale = sp.simplify(-sp.Poly(base, y_var).LC() * y0)
        coefficient = sp.simplify(sp.limit(term * base**power, y_var, y0) / scale**power)
        
        if power == 1:
            return sp.simplify(coefficient * a**n_var * u)
        elif power == 2:
            return sp.simplify(coefficient * (n_var + 1) * a**n_var * u)
            
    return term

def inverse_z_transform(expr, y_var, n_var):
    expr = sp.expand(expr)
    if isinstance(expr, sp.Add):
        return sp.simplify(sum(inverse_z_transform(t, y_var, n_var) for t in expr.args))
    return table_lookup_inverse_z(expr, y_var, n_var)

x_n_raw = inverse_z_transform(pfe_y, y, n)
print("\nInverse Z-Transform result x[n]:")
display(x_n_raw)

x_n = sp.factor(sp.simplify(x_n_raw))
print("\nSimplified Final Signal x[n]:")
display(x_n)

x_n_expected = (sp.Rational(3, 2) * n + 1) * (sp.Rational(1, 2)**n) * sp.Heaviside(n)
print("\nEquivalent expected form:")
display(x_n_expected)